In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
SHOW CATALOGS;

In [0]:
CREATE WIDGET TEXT storageName    DEFAULT "smartdatadatalake";
CREATE WIDGET TEXT catalogo       DEFAULT "project_dev";
CREATE WIDGET TEXT schemaBronze   DEFAULT "instacart_bronze";
CREATE WIDGET TEXT schemaSilver   DEFAULT "instacart_silver";
CREATE WIDGET TEXT schemaGold     DEFAULT "instacart_gold";
CREATE WIDGET TEXT credential_    DEFAULT "credential_dev";
 

In [0]:
create catalog if not exists ${catalogo};

In [0]:

create schema if not exists $catalogo.$schemaBronze;
create schema if not exists $catalogo.$schemaSilver;
create schema if not exists $catalogo.$schemaGold;
CREATE CATALOG IF NOT EXISTS ${catalogo};
create schema if not exists ${catalogo}.${schemaBronze};
create schema if not exists ${catalogo}.${schemaSilver};
create schema if not exists ${catalogo}.${schemaGold};

create catalog if not exists ${catalogo};
create schema 

In [0]:
CREATE SCHEMA IF NOT EXISTS nstacart_dev.raw;
CREATE SCHEMA IF NOT EXISTS nstacart_dev.bronze;
CREATE SCHEMA IF NOT EXISTS nstacart_dev.silver;
CREATE SCHEMA IF NOT EXISTS nstacart_dev.golden;
CREATE SCHEMA IF NOT EXISTS nstacart_dev.exploratory;

CREATE VOLUME IF NOT EXISTS nstacart_dev.raw.datasets;

In [0]:
schemaBronze = dbutils.widgets.get("schemaBronze")
schemaSilver = dbutils.widgets.get("schemaSilver")
schemaGold   = dbutils.widgets.get("schemaGold")
 
for schema in [schemaBronze, schemaSilver, schemaGold]:
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema}")
        print(f"  ✅ Schema {catalogo}.{schema} listo")
    except Exception as e:
        print(f"  ❌ Error creando {catalogo}.{schema}: {e}")
        raise

In [0]:

ddl_bronze = {
    "orders": f"""
        CREATE TABLE IF NOT EXISTS {catalogo}.{schemaBronze}.orders (
            order_id BIGINT,
            user_id BIGINT,
            eval_set STRING,
            order_number INT,
            order_dow INT,
            order_hour_of_day INT,
            days_since_prior_order DOUBLE,
            _ingestion_timestamp TIMESTAMP,
            _source_file STRING
        )
        USING DELTA
        LOCATION '{base_bronze}/orders'
    """,
    "order_products_prior": f"""
        CREATE TABLE IF NOT EXISTS {catalogo}.{schemaBronze}.order_products_prior (
            order_id BIGINT,
            product_id BIGINT,
            add_to_cart_order INT,
            reordered INT,
            _ingestion_timestamp TIMESTAMP,
            _source_file STRING
        )
        USING DELTA
        LOCATION '{base_bronze}/order_products_prior'
    """,
    "order_products_train": f"""
        CREATE TABLE IF NOT EXISTS {catalogo}.{schemaBronze}.order_products_train (
            order_id BIGINT,
            product_id BIGINT,
            add_to_cart_order INT,
            reordered INT,
            _ingestion_timestamp TIMESTAMP,
            _source_file STRING
        )
        USING DELTA
        LOCATION '{base_bronze}/order_products_train'
    """,
    "products": f"""
        CREATE TABLE IF NOT EXISTS {catalogo}.{schemaBronze}.products (
            product_id BIGINT,
            product_name STRING,
            aisle_id INT,
            department_id INT,
            _ingestion_timestamp TIMESTAMP,
            _source_file STRING
        )
        USING DELTA
        LOCATION '{base_bronze}/products'
    """,
    "aisles": f"""
        CREATE TABLE IF NOT EXISTS {catalogo}.{schemaBronze}.aisles (
            aisle_id INT,
            aisle STRING,
            _ingestion_timestamp TIMESTAMP,
            _source_file STRING
        )
        USING DELTA
        LOCATION '{base_bronze}/aisles'
    """,
    "departments": f"""
        CREATE TABLE IF NOT EXISTS {catalogo}.{schemaBronze}.departments (
            department_id INT,
            department STRING,
            _ingestion_timestamp TIMESTAMP,
            _source_file STRING
        )
        USING DELTA
        LOCATION '{base_bronze}/departments'
    """,
}
 
for table_name, ddl in ddl_bronze.items():
    try:
        spark.sql(ddl)
        print(f"  ✅ Tabla bronze creada: {catalogo}.{schemaBronze}.{table_name}")
    except Exception as e:
        print(f"  ⚠️ {table_name}: {e}")